In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.patches import Polygon
import seaborn as sns
import sys
sys.path.append("/home/jg2447/slayman/motif_inference/MOBI/scripts/")
from src import plt_func
from src import stats
from src import mobi

In [ ]:
enrichment_DNAinput = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/worm/enrichment/RankSPP_100.txt", sep="\t", index_col=0).loc[:,["250"]]
enrichment_DNAinput_c = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/worm/enrichment/RankCrowdness_100.txt", sep="\t", index_col=0).loc[:,["250"]]
enrichment_mockIP = pd.read_csv("/home/jg2447/slayman/motif_inference/result/mockIP/worm/enrichment/RankSPP_100.txt", sep="\t", index_col=0).loc[:,["250"]]
enrichment_mockIP_c = pd.read_csv("/home/jg2447/slayman/motif_inference/result/mockIP/worm/enrichment/RankCrowdness_100.txt", sep="\t", index_col=0).loc[:,["250"]]
inference_DNAinput = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/worm/tomtom_summary/DREME_RankSPP_100_top5.txt", sep="\t").loc[:,["TF", "accuracy"]]
inference_DNAinput_c = pd.read_csv("/home/jg2447/slayman/motif_inference/result/UnifyParaSearch/worm/tomtom_summary/DREME_RankCrowdness_100_top5.txt", sep="\t").loc[:,["TF", "accuracy"]]
inference_mockIP = pd.read_csv("/home/jg2447/slayman/motif_inference/result/mockIP/worm/tomtom_summary/DREME_RankSPP_100_top5.txt", sep="\t").loc[:,["TF", "accuracy"]]
inference_mockIP_c = pd.read_csv("/home/jg2447/slayman/motif_inference/result/mockIP/worm/tomtom_summary/DREME_RankCrowdness_100_top5.txt", sep="\t").loc[:,["TF", "accuracy"]]

enrichment_worm = enrichment_DNAinput
for df in [enrichment_mockIP, enrichment_DNAinput_c, enrichment_mockIP_c]:
    enrichment_worm = pd.merge(enrichment_worm, df, left_index=True, right_index=True)
inference_worm = inference_DNAinput
for df in [inference_mockIP, inference_DNAinput_c, inference_mockIP_c]:
    inference_worm = pd.merge(inference_worm, df, left_on="TF", right_on="TF")
inference_worm = inference_worm.set_index("TF")

worm_data = pd.merge(enrichment_worm, inference_worm, left_index=True, right_index=True)
worm_data.columns = ["enrichment_DNAinput", "enrichment_mockIP", "enrichment_DNAinput_c", "enrichment_mockIP_c", "inference_DNAinput", "inference_mockIP", "inference_DNAinput_c", "inference_mockIP_c"]

In [ ]:
def get_inference_data_p(tomtom_summary_dir, nbp_list, rank_list, tool, top):

    # the standard to compare for significant test
    standard_spp = pd.read_csv("%s/%s_RankSPP_100_%s.txt" % (tomtom_summary_dir, tool, top), sep="\t")["accuracy"].values

    nn_data = []
    for nn in nbp_list:
        rank_data = []
        for rank in rank_list:
            data = pd.read_csv("%s/%s_%s_%d_%s.txt" % (tomtom_summary_dir, tool, rank, nn, top), sep="\t")["accuracy"].values
            p = stats.sig_test_wilcoxon(data, standard_spp)
            rank_data.append(p)
        nn_data.append(rank_data)
    
    df_p_spp = pd.DataFrame(nn_data)
    df_p_spp.columns = rank_list
    df_p_spp.index = nbp_list

    # get sig star
    df_star = df_p_spp.copy().astype(str)
    df_star[df_p_spp < 0.05] = "*"
    df_star[df_p_spp >= 0.05] = ""
    
    return(df_star)

In [ ]:
alpha_list_para = np.round(np.append(np.arange(0.1,1.0,0.1), np.arange(1.0,11.0,1.0)), decimals=2)
rank_list_para = ["RankLinear_%.1f" % i for i in alpha_list_para]
rank_list = ["RankSPP"]
rank_list.extend(rank_list_para)
rank_list.extend(["RankCrowdness"])

nbp_list = [10, 20, 40, 60, 80, 100, 120, 140, 160, 180, 200]

tool = "DREME"
top = "top5"

df_fly = mobi.get_tomtom_summary_data("/home/jg2447/slayman/motif_inference/result/mockIP/fly/tomtom_summary/", nbp_list, rank_list, tool, top, improve=True)
df_fly_p = get_inference_data_p("/home/jg2447/slayman/motif_inference/result/mockIP/fly/tomtom_summary/", nbp_list, rank_list, tool, top)
df_fly.to_csv("./fig7_data_fly.txt", sep="\t")
df_fly_p.to_csv("./fig7_data_fly_p.txt", sep="\t")

df_worm = mobi.get_tomtom_summary_data("/home/jg2447/slayman/motif_inference/result/mockIP/worm/tomtom_summary/", nbp_list, rank_list, tool, top, improve=True)
df_worm_p = get_inference_data_p("/home/jg2447/slayman/motif_inference/result/mockIP/worm/tomtom_summary/", nbp_list, rank_list, tool, top)
df_worm.to_csv("./fig7_data_worm.txt", sep="\t")
df_worm_p.to_csv("./fig7_data_worm_p.txt", sep="\t")

In [ ]:
from scipy.stats import wilcoxon
wilcoxon(
    worm_data['inference_DNAinput_c'].values,
    worm_data['inference_mockIP_c'].values,
    alternative='less')

### plot

In [ ]:
def plt_heatmap(data, data_p, ax, ax_cbar, vmax):

    ax = sns.heatmap(
        data, 
        ax=ax, 
        cbar_ax=ax_cbar,
        annot=data_p,
        vmin=-vmax, vmax=vmax, center=0, cmap=cmap, annot_kws={"fontsize":5}, fmt="s", linewidths=.05)

    ax.get_xaxis().set_ticks(np.arange(0.5,11,1))
    ax.get_xaxis().set_ticklabels(
        np.array(data.columns)*2,
        rotation=90, va='center', ha='right', rotation_mode="anchor", fontsize=x_tick_label_fs)
    ax.get_xaxis().set_tick_params(size=2, width=0.5, pad=1)
    ax.set_xlabel("Binding site length", fontsize=x_label_fs)

    plt_func.rm_axis_label(ax, "y")
    return(ax)

In [ ]:
figsize = (4.7, 2.7)
panel_number_fs = 8
x_tick_label_fs = 5
y_tick_label_fs = 5
x_label_fs = 5
y_label_fs = 5
title_fs = 6
legend_fs = 5

fig, axs = plt.subplots(
    nrows=1, ncols=2, 
    sharex=False, sharey=False,
    gridspec_kw={'wspace': 0.1, "width_ratios":[1.5, 3.2]},
    figsize=figsize, dpi=300)

axs[0].axis('off')
axs[1].axis('off')

###############
gs = mpl.gridspec.GridSpecFromSubplotSpec(2, 2, subplot_spec=axs[0], wspace=0.6, hspace=0.3)

# A
ax = fig.add_subplot(gs[0, 0])
ax.bar([0,0.7], [worm_data["enrichment_DNAinput"].mean(), worm_data["enrichment_mockIP"].mean()], 
       width=0.5, color = ["#ed6a6d", "#ffffff"], edgecolor="#ed6a6d")
ax.set_xlim([-0.5, 1.25])
ax.xaxis.set_ticks([])
ax.xaxis.set_tick_params(size=0)   

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines["left"].set_linewidth(0.5)
ax.yaxis.set_ticks([0, 0.1, 0.2])
ax.yaxis.set_tick_params(size=2, pad=1, labelsize=x_tick_label_fs)   
plt_func.ax_trim(ax)
ax.set_ylabel("Motif enrichment", fontsize=x_label_fs, labelpad=0.3)

legend_elements = [Patch(facecolor='grey', edgecolor='grey',label='DNA input'),
                   Patch(facecolor='white', edgecolor='grey',label='Mock IP')]
ax.legend(handles=legend_elements, fontsize=legend_fs, ncol=2, frameon=False, bbox_to_anchor=(2.75,1.25), handletextpad=0.5, columnspacing=1)


ax.text(-0.6, 1, "A", transform=ax.transAxes, size=panel_number_fs, weight='bold')
ax.annotate(
        "Strong binding sites", 
        xy=(0.55, -0.15), 
        rotation=0, xycoords="axes fraction", fontsize=y_label_fs)


ax = fig.add_subplot(gs[0, 1])
ax.bar([0,0.7], [worm_data["inference_DNAinput"].mean(), worm_data["inference_mockIP"].mean()], 
       width=0.5, color = ["#ed6a6d", "#ffffff"], edgecolor="#ed6a6d")
ax.set_xlim([-0.5, 1.25])
ax.xaxis.set_ticks([])
ax.xaxis.set_tick_params(size=0)   

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines["left"].set_linewidth(0.5)
ax.yaxis.set_ticks([0, 0.35, 0.7])
ax.yaxis.set_tick_params(size=2, pad=1, labelsize=x_tick_label_fs)   
plt_func.ax_trim(ax)
ax.set_ylabel("Correct inference", fontsize=x_label_fs, labelpad=0.3)

ax = fig.add_subplot(gs[1, 0])
ax.bar([0,0.7], [worm_data["enrichment_DNAinput_c"].mean(), worm_data["enrichment_mockIP_c"].mean()], 
       width=0.5, color = ["#c2d2ed", "#ffffff"], edgecolor="#c2d2ed")
ax.set_xlim([-0.5, 1.25])
ax.xaxis.set_ticks([])
ax.xaxis.set_tick_params(size=0)   

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines["left"].set_linewidth(0.5)
ax.yaxis.set_ticks([0, 0.1, 0.2])
ax.yaxis.set_tick_params(size=2, pad=1, labelsize=x_tick_label_fs)   
plt_func.ax_trim(ax)
ax.set_ylabel("Motif enrichment", fontsize=x_label_fs, labelpad=0.3)

ax.text(-0.6, 1, "B", transform=ax.transAxes, size=panel_number_fs, weight='bold')
ax.annotate(
        "Sites with low C-scores", 
        xy=(0.4, -0.15), 
        rotation=0, xycoords="axes fraction", fontsize=y_label_fs)


ax = fig.add_subplot(gs[1, 1])
ax.bar([0,0.7], [worm_data["inference_DNAinput_c"].mean(), worm_data["inference_mockIP_c"].mean()], 
       width=0.5, color = ["#c2d2ed", "#ffffff"], edgecolor="#c2d2ed")
ax.set_xlim([-0.5, 1.25])
ax.xaxis.set_ticks([])
ax.xaxis.set_tick_params(size=0)   

ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines["left"].set_linewidth(0.5)
ax.yaxis.set_ticks([0, 0.35, 0.7])
ax.yaxis.set_tick_params(size=2, pad=1, labelsize=x_tick_label_fs)   
plt_func.ax_trim(ax)
ax.set_ylabel("Correct inference", fontsize=x_label_fs, labelpad=0.3)


###############

cmap = "RdBu"

# read data
df_fly = pd.read_csv("./fig7_data_fly.txt", sep="\t", index_col=0).T
df_fly_p = pd.read_table("./fig7_data_fly_p.txt", sep="\t", index_col=0, na_filter=False).T
df_worm = pd.read_csv("./fig7_data_worm.txt", sep="\t", index_col=0).T
df_worm_p = pd.read_table("./fig7_data_worm_p.txt", sep="\t", index_col=0, na_filter=False).T

vmax = df_fly.abs().max().max()

gs = mpl.gridspec.GridSpecFromSubplotSpec(3, 6, subplot_spec=axs[1], width_ratios=[3, 11, 0.5, 0.15, 11, 0.5], wspace=0.15)

ax_cbar = fig.add_subplot(gs[1, 2])

#### C
ax = fig.add_subplot(gs[:, 1])

plt_heatmap(df_fly, df_fly_p, ax, ax_cbar, vmax)
ax.set_title("Fly", fontsize=title_fs)
ax.annotate("★", (5.05, 0.8), fontsize=6, color='red')
ax.text(-0.2, 1.05, "C", transform=ax.transAxes, size=panel_number_fs, weight='bold')

ax.get_yaxis().set_ticks(np.arange(0.5,21,1))
ax.get_yaxis().set_ticklabels(["Signal","0.1", "0.2", "0.3", "0.4", "0.5", "0.6", "0.7", "0.8", "0.9", "1.0", "2.0", "3.0", "4.0", "5.0", "6.0", "7.0", "8.0", "9.0", "10.0","C-score"],  
                              fontsize=y_tick_label_fs)
ax.get_yaxis().set_tick_params(size=2, width=0.5, pad=1)
ax.set_ylabel("Binding site ranking method", fontsize=y_label_fs, labelpad=-0.2)

####
# color bar ticks
plt_func.rm_axis_label(ax_cbar, "y")
ax_cbar.text(-1.5, vmax+0.05, "%.2f" % vmax, fontsize=4)
ax_cbar.text(-1.5, -vmax-0.1, "-%.2f" % vmax, fontsize=4)


##### BC-score label
ax_annot = fig.add_subplot(gs[:, 0])
ax_annot.plot([1,1], [0,21], ",", ms=0)
ax_annot.set_ylim([0,21])
ax_annot.set_xlim([0,2])    
ax_annot.annotate("", 
                  (0.4, 1.5),
                  (0.4, 4),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (0.4, 17),
                  (0.4, 19.5),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (0, 1.5),
                  (0.8, 1.5),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("", 
                  (0, 19.5),
                  (0.8, 19.5),
                  arrowprops=dict(arrowstyle="-", fc="k", ec="k", lw=0.5))
ax_annot.annotate("BC-score with different weight", 
                  (0.2, 4.1),
                  (0.2, 4.1),
                  rotation=90,
                  size=6)
ax_annot.axis('off')

#### D

ax_cbar = fig.add_subplot(gs[1, 5])
vmax = df_worm.abs().max().max()

ax = fig.add_subplot(gs[:, 4])
plt_heatmap(df_worm, df_worm_p, ax, ax_cbar, vmax)
ax.set_title("Worm", fontsize=title_fs)
ax.annotate("★", (5.05, 0.8), fontsize=6, color='red')
ax.text(-0.1, 1.05, "D", transform=ax.transAxes, size=panel_number_fs, weight='bold')

####
# color bar ticks
plt_func.rm_axis_label(ax_cbar, "y")
ax_cbar.text(-4, vmax+0.15, "%.2f" % vmax, fontsize=4)
ax_cbar.text(-4, -vmax-0.3, "-%.2f" % vmax, fontsize=4)


plt.savefig("./fig7.pdf", dpi="figure", bbox_inches="tight")